# Final Model 2: 모델링 강화 버전 (Ensemble)

이 노트북은 `submission_holiday_dinner_stronger_annotated.ipynb`를 기반으로 **모델링(Modeling)**을 강화한 버전입니다.

**추가된 내용:**
1. **다양한 모델 도입:** XGBoost 외에 **CatBoost**, **LightGBM**을 추가로 사용합니다.
2. **앙상블 (Ensemble):** 3가지 모델의 예측값을 평균내어 일반화 성능을 높입니다.

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# 경로 자동 탐색 및 데이터 로드 (기존과 동일)
train_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\train_median.csv",
    r"./data/train_median.csv",
    r"../data/train_median.csv",
]
test_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\test.csv",
    r"./data/test.csv",
    r"../data/test.csv",
]
sample_sub_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\sample_submission.csv",
    r"./data/sample_submission.csv",
    r"../data/sample_submission.csv",
]
weather_candidates = [
    r"C:\Users\CY2\github\SecondProject\data\weather.csv",
    r"./data/weather.csv",
    r"../data/weather.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
test_path = find_existing_path(test_candidates)
sample_sub_path = find_existing_path(sample_sub_candidates)
weather_path = find_existing_path(weather_candidates)

if train_path is None or test_path is None or sample_sub_path is None or weather_path is None:
    train = pd.read_csv("train_median.csv", encoding="utf-8-sig")
    test = pd.read_csv("test.csv", encoding="utf-8-sig")
    sample_submission = pd.read_csv("sample_submission.csv", encoding="utf-8-sig")
    weather = pd.read_csv("weather.csv", encoding="utf-8-sig")
else:
    train = pd.read_csv(train_path, encoding="utf-8-sig")
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    sample_submission = pd.read_csv(sample_sub_path, encoding="utf-8-sig")
    weather = pd.read_csv(weather_path, encoding="utf-8-sig")

train["일자"] = pd.to_datetime(train["일자"])
test["일자"] = pd.to_datetime(test["일자"])

In [ ]:
# 날씨 전처리 (기존과 동일)
def find_col(cols, candidates):
    for cand in candidates:
        for c in cols:
            if cand.lower() == c.lower():
                return c
        for c in cols:
            if cand.lower() in c.lower():
                return c
    return None

date_col = find_col(weather.columns, ["일시", "date", "날짜"])
temp_col = find_col(weather.columns, ["기온", "평균기온", "temperature", "avg_temp"])
rain_col = find_col(weather.columns, ["강수량", "rainfall", "precipitation", "rain"])

weather = weather[[date_col, temp_col, rain_col]].copy()
weather = weather.rename(columns={date_col:"일자", temp_col:"기온", rain_col:"강수량"})
weather["일자"] = pd.to_datetime(weather["일자"])
weather["기온"] = pd.to_numeric(weather["기온"], errors="coerce").interpolate().bfill().ffill()
weather["강수량"] = pd.to_numeric(weather["강수량"], errors="coerce").fillna(0)
weather["is_rain"] = (weather["강수량"] > 0).astype(int)
weather["is_hot"] = (weather["기온"] >= 28).astype(int)
weather["is_cold"] = (weather["기온"] <= 5).astype(int)

train = train.merge(weather, on="일자", how="left")
test = test.merge(weather, on="일자", how="left")

for df_ in [train, test]:
    for c in ["기온","강수량","is_rain","is_hot","is_cold"]:
        if c.startswith("is_"):
            df_[c] = df_[c].fillna(0)
        else:
            df_[c] = df_[c].interpolate().bfill().ffill()

In [ ]:
# 피처 생성 (기존과 동일)
def add_features(df):
    df = df.copy().sort_values("일자").reset_index(drop=True)
    df["월"] = df["일자"].dt.month
    df["일"] = df["일자"].dt.day
    df["요일"] = df["일자"].dt.weekday
    df["식사가능자수"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=1)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["요일"] == 2).astype(int)
    df["is_fri"] = (df["요일"] == 4).astype(int)
    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)
    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])

    prev_gap = df["일자"].diff().dt.days.fillna(1)
    next_gap = df["일자"].shift(-1).sub(df["일자"]).dt.days.fillna(1)
    df["holiday_after"] = (prev_gap >= 2).astype(int)
    df["holiday_before"] = (next_gap >= 2).astype(int)

    return df

train = add_features(train)
test = add_features(test)

train["중식참여율"] = (train["중식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)
train["석식참여율"] = (train["석식계"] / train["식사가능자수"]).clip(lower=0, upper=1.5)

lunch_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_fri","days_to_month_end","is_month_end_part",
]
dinner_features = [
    "월","일","요일","식사가능자수","본사출장자수","본사시간외근무명령서승인건수",
    "is_wed","has_overtime","log_overtime",
]

In [ ]:
# [추가] 앙상블 학습 (XGB + Cat + LGBM)

X_train_lunch = train[lunch_features]
X_test_lunch = test[lunch_features]
y_lunch = train["중식참여율"]
X_train_dinner = train[dinner_features]
X_test_dinner = test[dinner_features]
y_dinner = train["석식참여율"]

# 1. XGBoost
xgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "min_child_weight": 1,
    "random_state": 42,
    "n_jobs": -1
}
xgb_lunch = XGBRegressor(**xgb_params)
xgb_dinner = XGBRegressor(**xgb_params)
xgb_lunch.fit(X_train_lunch, y_lunch)
xgb_dinner.fit(X_train_dinner, y_dinner)

# 2. CatBoost
cat_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "verbose": 0,
    "random_state": 42,
    "allow_writing_files": False
}
cat_lunch = CatBoostRegressor(**cat_params)
cat_dinner = CatBoostRegressor(**cat_params)
cat_lunch.fit(X_train_lunch, y_lunch)
cat_dinner.fit(X_train_dinner, y_dinner)

# 3. LightGBM
lgbm_params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 5,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1
}
lgbm_lunch = LGBMRegressor(**lgbm_params)
lgbm_dinner = LGBMRegressor(**lgbm_params)
lgbm_lunch.fit(X_train_lunch, y_lunch)
lgbm_dinner.fit(X_train_dinner, y_dinner)

# 예측 평균 (Simple Averaging)
pred_lunch_xgb = xgb_lunch.predict(X_test_lunch)
pred_lunch_cat = cat_lunch.predict(X_test_lunch)
pred_lunch_lgbm = lgbm_lunch.predict(X_test_lunch)
base_pred_lunch_ratio = (pred_lunch_xgb + pred_lunch_cat + pred_lunch_lgbm) / 3

pred_dinner_xgb = xgb_dinner.predict(X_test_dinner)
pred_dinner_cat = cat_dinner.predict(X_test_dinner)
pred_dinner_lgbm = lgbm_dinner.predict(X_test_dinner)
base_pred_dinner_ratio = (pred_dinner_xgb + pred_dinner_cat + pred_dinner_lgbm) / 3

base_pred_lunch_ratio = np.clip(base_pred_lunch_ratio, 0, 1.5)
base_pred_dinner_ratio = np.clip(base_pred_dinner_ratio, 0, 1.5)

In [ ]:
# Weak Corrections (기존 유지)
train_lunch_menu = pd.DataFrame([{"kw_dummy":0} for _ in range(len(train))]) # 메뉴 보정 생략을 위한 더미
test_lunch_menu = pd.DataFrame([{"kw_dummy":0} for _ in range(len(test))])
train_dinner_menu = pd.DataFrame([{"kw_dummy":0} for _ in range(len(train))])
test_dinner_menu = pd.DataFrame([{"kw_dummy":0} for _ in range(len(test))])

# 메뉴 Correction을 제외하고 날씨/연휴 Correction만 적용
weather_lunch_signal = (
    0.010 * test["is_rain"].values
    - 0.006 * test["is_hot"].values
    + 0.004 * test["is_cold"].values
)
weather_dinner_signal = (
    0.004 * test["is_rain"].values
    + 0.003 * test["is_cold"].values
)

holiday_lunch_signal = (
    -0.004 * test["holiday_before"].values
    + 0.003 * test["holiday_after"].values
)
holiday_dinner_signal = (
    -0.005 * test["holiday_before"].values
    + 0.003 * test["holiday_after"].values
)

final_pred_lunch_ratio = np.clip(base_pred_lunch_ratio + weather_lunch_signal + holiday_lunch_signal, 0, 1.5)
final_pred_dinner_ratio = np.clip(base_pred_dinner_ratio + weather_dinner_signal + holiday_dinner_signal, 0, 1.5)

pred_lunch_count = np.clip(final_pred_lunch_ratio * test["식사가능자수"].values, 0, None)
pred_dinner_count = np.clip(final_pred_dinner_ratio * test["식사가능자수"].values, 0, None)

submission = sample_submission.copy()
if "중식계" in submission.columns:
    submission["중식계"] = pred_lunch_count
    submission["석식계"] = pred_dinner_count
else:
    submission.iloc[:, 1] = pred_lunch_count
    submission.iloc[:, 2] = pred_dinner_count

submission.to_csv("finalmodel2_submission.csv", index=False, encoding="utf-8-sig")
print("저장 완료: finalmodel2_submission.csv")